# Experiment 5 — Full Context + Hierarchy

Full Theme + Epic + Stage context. Candidate L3s include ID, name, description, tier, Level 1 name, and Level 2 name.

In [ ]:
from pathlib import Path
import ast, json, os
import httpx, pandas as pd
from IPython.display import display
from common import load_gateway, call_llm, parse_json_response, validate_l3_response, score_sets, summarize_results, save_results_excel
THEME_IDS=None; VALUE_STREAM_STAGE_FIELD_ID="customfield_18700"
def clean(v): return "" if v is None or (not isinstance(v,(list,dict)) and pd.isna(v)) else str(v).strip()
def parse_list(v):
    if v is None or pd.isna(v): return []
    try:
        x=ast.literal_eval(str(v)); return [clean(i) for i in x] if isinstance(x,(list,tuple,set)) else [clean(x)]
    except Exception: return [x.strip() for x in str(v).strip("[]").split(",") if x.strip()]
def roots(): return [Path.cwd(),Path.cwd().parent]
def find_name(name):
    for r in roots():
        p=r/name
        if p.exists(): return p
        h=list(r.rglob(name))
        if h: return h[0]
    raise FileNotFoundError(name)
def read_table(p):
    p=Path(p); return pd.read_csv(p,dtype=str,encoding="cp1252",encoding_errors="replace") if p.suffix.lower()==".csv" else pd.read_excel(p,dtype=str)
def find_cols(cols):
    req=set(cols)
    for r in roots():
        for p in list(r.rglob("*.csv"))+list(r.rglob("*.xlsx")):
            try:
                if req.issubset(read_table(p).columns): return p
            except Exception: pass
    raise FileNotFoundError(cols)
def load_themes(ids=None):
    df=pd.read_csv(find_name("epic_gen.csv"),dtype=str,encoding="cp1252",encoding_errors="replace"); df=df if not ids else df[df["key"].isin(ids)]; out={}
    for _,r in df.iterrows():
        keys=parse_list(r.get("epic_keys")); d=parse_list(r.get("epic_description")); s=parse_list(r.get("epic_successCriteria")); out[clean(r["key"])]=dict(theme_description=clean(r.get("description")),theme_business_needs=clean(r.get("businessNeeds")),epics=[dict(key=k,description=d[i] if i<len(d) else "",success_criteria=s[i] if i<len(s) else "") for i,k in enumerate(keys)])
    return out
def jira_headers():
    if os.getenv("JIRA_HEADERS_JSON"): return json.loads(os.environ["JIRA_HEADERS_JSON"])
    if os.getenv("JIRA_BEARER_TOKEN"): return {"Authorization":f"Bearer {os.environ['JIRA_BEARER_TOKEN']}","Accept":"application/json"}
    raise RuntimeError("Set Jira auth env vars")
def epic_stage_ids(key):
    base=os.environ["JIRA_BASE_URL"].rstrip("/")
    with httpx.Client(headers=jira_headers(),verify=os.getenv("JIRA_VERIFY_SSL","false").lower()=="true",timeout=30) as c: r=c.get(f"{base}/rest/api/2/issue/{key}",params={"fields":VALUE_STREAM_STAGE_FIELD_ID}); r.raise_for_status()
    raw=r.json().get("fields",{}).get(VALUE_STREAM_STAGE_FIELD_ID) or []; raw=raw if isinstance(raw,list) else [raw]; return [clean(x.get("id") or x.get("value")) if isinstance(x,dict) else clean(x) for x in raw if x]
STAGES=read_table(find_cols(["Value Stream Stage ID","Value Stream Stage Name","Value Stream Stage Description","Value Stream Stage Entrance Criteria","Value Stream Stage Exit Criteria"])); MAP=read_table(find_cols(["Value Stream Stage ID","Capability ID","Capability Name","Level 1 Name","Level 2 Name"])); MASTER=read_table(find_cols(["Capability ID","Capability Name","Capability Description","Capability Tier"])); GT=read_table(find_cols(["epic_key","l3_capability_id"]))
def stage_context(s):
    r=STAGES[STAGES["Value Stream Stage ID"].astype(str).str.strip()==s].iloc[0]; return {"stage_id":s,"stage_name":clean(r["Value Stream Stage Name"]),"stage_description":clean(r["Value Stream Stage Description"]),"entrance_criteria":clean(r["Value Stream Stage Entrance Criteria"]),"exit_criteria":clean(r["Value Stream Stage Exit Criteria"])}
def candidates(s,hierarchy=False):
    m=MAP[MAP["Value Stream Stage ID"].astype(str).str.strip()==s].merge(MASTER[["Capability ID","Capability Description","Capability Tier"]],on="Capability ID",how="left"); out=[]
    for _,r in m.iterrows():
        x={"capability_id":clean(r["Capability ID"]),"capability_name":clean(r["Capability Name"]),"capability_description":clean(r["Capability Description"]),"capability_tier":clean(r["Capability Tier"])}
        if hierarchy: x.update(level_1_name=clean(r["Level 1 Name"]),level_2_name=clean(r["Level 2 Name"]))
        out.append(x)
    return out
def gt_map(): return {k:set(g["l3_capability_id"].dropna().astype(str).str.strip()) for k,g in GT.groupby("epic_key")}

In [ ]:
EXPERIMENT_NAME="E5_FULL_WITH_HIERARCHY"
SEND_THEME_BUSINESS_NEEDS=True
SEND_THEME_DESCRIPTION=True
SEND_EPIC_DESCRIPTION=True
SEND_SUCCESS_CRITERIA=True
SEND_HIERARCHY=True

## Production prompt

In [ ]:
SYSTEM_PROMPT = 'You are a Business Capability Architecture specialist mapping context to enterprise Level 3 (L3) business capabilities.\n\nSelect only from candidate_l3_capabilities. The candidate L3 name and description define the business function; tier and hierarchy are supporting taxonomy context only. When Epic evidence is supplied, treat it as primary. Theme context is broader strategic context. Value Stream Stage is process context and narrows the candidate set but is not sufficient by itself. Perform semantic mapping, not keyword matching. Prefer the smallest defensible set. Do not select adjacent, upstream, downstream, stakeholder, data-provider, or merely technical capabilities. Never invent or modify capability IDs. If no candidate is sufficiently supported, return an empty l3 list. Select at most three.\n\nReturn JSON only: {"l3":[{"capability_id":"CAP00000000","reason":"Specific evidence-based explanation."}]}'
def build_user_prompt(theme,epic,stage,cand):
    p={"task":"Select the most appropriate L3 business capabilities from the supplied candidates.","theme":{"business_needs":theme["theme_business_needs"],"description":theme["theme_description"]},"epic":{"description":epic["description"],"success_criteria":epic["success_criteria"]},"value_stream_stage":stage,"candidate_l3_capabilities":cand,"selection_instruction":"Select 0 to 3 L3 capabilities; return [] when none is sufficiently supported."}; return json.dumps(p,ensure_ascii=False,indent=2)

## Run and evaluate

In [ ]:
def run():
    gateway=load_gateway(); truth=gt_map(); rows=[]
    for theme_id,theme in load_themes(THEME_IDS).items():
        for epic in theme["epics"]:
            predicted=set(); stage_runs=[]
            try:
                for sid in epic_stage_ids(epic["key"]):
                    st=stage_context(sid); cand=candidates(sid,SEND_HIERARCHY); selected=validate_l3_response(parse_json_response(call_llm(gateway,SYSTEM_PROMPT,build_user_prompt(theme,epic,st,cand))),[c["capability_id"] for c in cand],allow_empty=True,max_selected=3); predicted.update(x["capability_id"] for x in selected); stage_runs.append({"stage_id":sid,"selected":selected})
                t=truth.get(epic["key"]); metrics=score_sets(predicted,t) if t is not None else {"exact_match":None,"precision":None,"recall":None,"f1":None,"predicted_count":len(predicted),"truth_count":None}; rows.append({"experiment":EXPERIMENT_NAME,"theme_id":theme_id,"epic_key":epic["key"],"predicted_l3_ids":json.dumps(sorted(predicted)),"ground_truth_l3_ids":json.dumps(sorted(t)) if t is not None else None,"stage_predictions":json.dumps(stage_runs),"status":"ok" if t is not None else "missing_ground_truth","error":None,**metrics})
            except Exception as e: rows.append({"experiment":EXPERIMENT_NAME,"theme_id":theme_id,"epic_key":epic["key"],"status":"error","error":str(e),"exact_match":None,"precision":None,"recall":None,"f1":None})
    return pd.DataFrame(rows)
results=run(); display(summarize_results(results)); display(results.head(20)); save_results_excel(results,EXPERIMENT_NAME,"results")